In [24]:
import pandas as pd
import numpy as np
import os


DOSSIER_DONNEES = '../data'

SEPARATEUR = '|' 

fichiers_dvf = [
    'ValeursFoncieres-2020-S2.txt',
    'ValeursFoncieres-2021.txt',
    'ValeursFoncieres-2022.txt',
    'ValeursFoncieres-2023.txt',
    'ValeursFoncieres-2024.txt',
    'ValeursFoncieres-2025-S1.txt'
]

print("✅ Initialisation terminée.")

✅ Initialisation terminée.


In [25]:
liste_dfs = []

print(f"Début du chargement et de la fusion de {len(fichiers_dvf)} fichiers...")

for fichier in fichiers_dvf:
    chemin_complet = os.path.join(DOSSIER_DONNEES, fichier)
    
    try:
        # CORRECTION CLÉ : Utilisation de l'encodage 'latin-1' (ou 'ISO-8859-1')
        df_temp = pd.read_csv(chemin_complet, sep=SEPARATEUR, low_memory=False, encoding='latin-1')
        liste_dfs.append(df_temp)
        print(f"   - Chargé : {fichier} ({len(df_temp):,} lignes)")
    except FileNotFoundError:
        print(f"   - ❌ ERREUR : Le fichier {chemin_complet} n'a pas été trouvé. Vérifiez le chemin : {os.getcwd()}")
    except Exception as e:
        print(f"   - ❌ ERREUR lors du chargement de {fichier} : {e}")

# Concaténation (Fusion)
if liste_dfs:
    df_foncier_brut = pd.concat(liste_dfs, ignore_index=True)
    print("\n---")
    print(f"Total de transactions brutes fusionnées : {len(df_foncier_brut):,}")

    # NETTOYAGE DES NOMS DE COLONNES (Étape cruciale contre la KeyError)
    # 1. Mise en minuscule
    df_foncier_brut.columns = df_foncier_brut.columns.str.lower()
    # 2. Remplacement des espaces par des underscores et suppression des accents
    df_foncier_brut.columns = df_foncier_brut.columns.str.replace(' ', '_', regex=False)
    df_foncier_brut.columns = df_foncier_brut.columns.str.replace('é', 'e', regex=False)
    df_foncier_brut.columns = df_foncier_brut.columns.str.replace('è', 'e', regex=False)
    df_foncier_brut.columns = df_foncier_brut.columns.str.replace('î', 'i', regex=False)
    # 3. Suppression des espaces en début/fin
    df_foncier_brut.columns = df_foncier_brut.columns.str.strip() 

    print("✅ Noms de colonnes standardisés (minuscule + underscore).")
    print("Aperçu des colonnes après nettoyage :")
    print(df_foncier_brut.columns[df_foncier_brut.columns.str.contains('mutation|valeur|local')].tolist())
else:
    print("\n---")
    print("❌ ATTENTION : La liste des DataFrames est vide. Aucun fichier n'a été chargé.")
    df_foncier_brut = pd.DataFrame()

Début du chargement et de la fusion de 6 fichiers...
   - Chargé : ValeursFoncieres-2020-S2.txt (2,065,003 lignes)
   - Chargé : ValeursFoncieres-2021.txt (4,674,176 lignes)
   - Chargé : ValeursFoncieres-2022.txt (4,675,007 lignes)
   - Chargé : ValeursFoncieres-2023.txt (3,812,327 lignes)
   - Chargé : ValeursFoncieres-2024.txt (3,489,149 lignes)
   - Chargé : ValeursFoncieres-2025-S1.txt (1,387,077 lignes)

---
Total de transactions brutes fusionnées : 20,102,739
✅ Noms de colonnes standardisés (minuscule + underscore).
Aperçu des colonnes après nettoyage :
['date_mutation', 'nature_mutation', 'valeur_fonciere', 'code_type_local', 'type_local', 'identifiant_local']


In [26]:
print("Début du nettoyage et du filtrage des données...")

# 3.1 Sélection des colonnes utiles pour Hugo (Noms corrigés basés sur la structure DVF)
# Utilisation de 'commune', 'type_local', 'surface_reelle_bati', 'nombre_pieces_principales'
colonnes_utiles = [
    'date_mutation', 
    'valeur_fonciere', 
    'commune', # <-- CORRECTION : Utilisation de 'commune' (plus probable que 'nom_commune')
    'type_local', # <-- CORRECTION : Utilisation de 'type_local'
    'surface_reelle_bati', # <-- Nom souvent plus simple
    'nombre_pieces_principales', # <-- CORRECTION : Sans le '_de_'
    'code_postal' 
]

# Vérifiez que le DataFrame n'est pas vide avant de continuer
if df_foncier_brut.empty:
    print("❌ ERREUR : DataFrame vide. Arrêt du nettoyage.")
    df_clean = pd.DataFrame()
else:
    # Changement des noms dans la liste 'colonnes_utiles' pour correspondre aux noms DVF simplifiés
    df_clean = df_foncier_brut[colonnes_utiles].copy()

    # 3.2 Nettoyage et conversion des types de données
    df_clean['valeur_fonciere'] = df_clean['valeur_fonciere'].astype(str).str.replace(',', '.', regex=False)
    df_clean['valeur_fonciere'] = pd.to_numeric(df_clean['valeur_fonciere'], errors='coerce')

    df_clean['date_mutation'] = pd.to_datetime(df_clean['date_mutation'], format='%d/%m/%Y', errors='coerce')

    # 3.3 Filtrage initial des données non exploitables
    colonnes_filtres = ['valeur_fonciere', 'surface_reelle_bati', 'commune', 'date_mutation'] # Ajustement des noms
    df_clean.dropna(subset=colonnes_filtres, inplace=True)
    df_clean = df_clean[df_clean['surface_reelle_bati'] > 10]

    # 3.4 Calcul de la variable clé : Prix au m²
    df_clean['prix_au_m2'] = df_clean['valeur_fonciere'] / df_clean['surface_reelle_bati']

    # 3.5 Ciblages spécifiques pour l'investisseur Hugo Martin (Location Courte Durée)
    # 1. Garder seulement les 'Appartement'
    # Utilisation de 'type_local'
    df_clean = df_clean[df_clean['type_local'] == 'Appartement']

    # 2. Créer le type de bien simplifié (T1, T2, T3)
    # Utilisation de 'nombre_pieces_principales'
    df_clean['type_de_bien'] = df_clean['nombre_pieces_principales'].apply(
        lambda x: f'T{int(x)}' if pd.notna(x) and x > 0 else 'Non_specifie'
    )
    df_clean = df_clean[df_clean['type_de_bien'].isin(['T1', 'T2', 'T3'])]

    # 3. Filtration des transactions extrêmes (prix trop bas ou trop hauts)
    df_clean = df_clean[
        (df_clean['prix_au_m2'] >= 500) & 
        (df_clean['prix_au_m2'] <= 10000) 
    ]

    # Nettoyage des colonnes devenues inutiles après le filtrage
    df_clean.drop(columns=['nombre_pieces_principales', 'type_local'], inplace=True)

    print(f"🎉 Nettoyage terminé. Transactions retenues : {len(df_clean):,}")
    print("Aperçu des données propres :")
    display(df_clean.head())

Début du nettoyage et du filtrage des données...
🎉 Nettoyage terminé. Transactions retenues : 1,816,784
Aperçu des données propres :


,date_mutation,valeur_fonciere,commune,surface_reelle_bati,code_postal,prix_au_m2,type_de_bien
41,2020-07-03,179970.0,SAINT-LAURENT-SUR-SAONE,61.0,1750.0,2950.327869,T3
45,2020-07-02,136000.0,BOURG-EN-BRESSE,62.0,1000.0,2193.548387,T3
54,2020-07-06,124000.0,MONTREVEL-EN-BRESSE,53.0,1340.0,2339.622642,T2
57,2020-07-01,270000.0,BOURG-EN-BRESSE,111.0,1000.0,2432.432432,T2
59,2020-07-01,270000.0,BOURG-EN-BRESSE,31.0,1000.0,8709.677419,T2


In [27]:
NOM_FICHIER_PROPRE = DOSSIER_DONNEES + '_DVF_clean.csv'

df_clean.to_csv(NOM_FICHIER_PROPRE, index=False)

print(f"\n✨ Fichier propre sauvegardé : {NOM_FICHIER_PROPRE}")
print("Ce fichier est prêt à être utilisé par vos notebooks d'exploration et de visualisation.")


✨ Fichier propre sauvegardé : ../data_DVF_clean.csv
Ce fichier est prêt à être utilisé par vos notebooks d'exploration et de visualisation.
